# 8 · RAG Reranker: the right passage was at rank 6

## The problem

You ask your docs bot *"What should I do if my sourdough starter isn't rising?"*. The retriever (BM25 or embeddings)
ranks passages by **surface similarity**, so passages that merely contain the words *rising* and *starter* (a sunrise,
a starter kit, starter homes) float to the top. The one passage that actually answers the question sits at **rank 6**.
The LLM only reads the top few, so it answers from the wrong context, confidently.

The standard fix is a **reranker**: a second, smarter pass that reads the query and each candidate together and
re-orders them by how well each one *answers* the query.

## Why Jev

Reranking is pure judgment: *"how well does this passage answer this query?"*, once per candidate. With Jev that is
one `Score` per passage, **all in one request**, in about 100 ms, for a fraction of a cent. And the scores are
**calibrated** positions on a rubric you wrote, so you can do more than sort: you can set a **threshold** and drop
the junk before it ever reaches the LLM.

## What we'll build

```
query + 8 retrieved passages ──► Jev: 8 Scores in ONE call ──► sort by score ──► keep score ≥ KEEP ──► LLM context
                                  "how well does passage i        (rerank)         (filter)
                                   answer the query?" 0..3
```

## What you'll learn
- How to ask **many parallel questions** in one Jev call, one per item in a list.
- How a `Score` rubric turns into a sortable, thresholdable relevance number.
- How to point an instruction at one element of a list inside the state (`` `passages[3]` ``).
- Why a reranker should also be a **filter**, and what to do when nothing passes.

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads `TYPESAFE_API_KEY` from the `.env` file at the repo root,
- creates the Jev client `jev`,
- imports `Score`, the only question type this notebook needs,
- defines `bar()`, a tiny text progress bar we use to draw scores, and `PRICE_PER_M`, Jev's price per million input
  tokens (output is free).

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Score

# Look for a .env file in this folder or any parent folder
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()   # Jev does the judging
PRICE_PER_M = 0.042      # dollars per 1M input tokens (output tokens are free)


def bar(p, width=24):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · The query and the retriever's (bad) ranking

`CANDIDATES` stands in for the top 8 results of a keyword retriever, **in the order the retriever returned them**.
Look at the keyword traps: three passages match *rising* or *starter* but have nothing to do with bread. The passage
that actually answers the question is **#6**.

In [2]:
QUERY = "What should I do if my sourdough starter isn't rising?"

CANDIDATES = [
    "The sun was rising over the bakery as the queue formed outside.",            # 'rising' trap
    "Our starter kit for new bakers includes three loaf tins and a scraper.",     # 'starter' trap
    "Bread has been a staple food across human civilizations for millennia.",     # on-topic-ish, useless
    "Rising interest rates have pushed up the cost of commercial ovens.",          # 'rising' trap
    "A sourdough starter is a fermented culture of flour and water.",              # relevant background
    "If your starter is sluggish, feed it more often and keep it around 24-26C; "
    "discard half before each feeding to concentrate the yeast.",                  # THE answer, at rank 6
    "Starter homes in the area now average four hundred thousand dollars.",        # 'starter' trap
    "Preheat a Dutch oven to 230C before baking for the best crust.",             # baking, not the question
]
for i, c in enumerate(CANDIDATES, 1):
    print(f"{i}. {c[:80]}")

1. The sun was rising over the bakery as the queue formed outside.
2. Our starter kit for new bakers includes three loaf tins and a scraper.
3. Bread has been a staple food across human civilizations for millennia.
4. Rising interest rates have pushed up the cost of commercial ovens.
5. A sourdough starter is a fermented culture of flour and water.
6. If your starter is sluggish, feed it more often and keep it around 24-26C; disca
7. Starter homes in the area now average four hundred thousand dollars.
8. Preheat a Dutch oven to 230C before baking for the best crust.


## Step 2 · One `Score` per passage, all in one call

**`Score`** is Jev's graded-scale question. `criteria` is an **ordered rubric**: the first line is level 0, the last is
level 3. Jev returns `.score`, a **continuous** position on that scale (2.6 means "between partially and directly
relevant, closer to directly"), plus a `.confidence`.

| Level | Meaning |
|---|---|
| 0 | Irrelevant: does not address the query at all |
| 1 | Tangential: same topic area but does not help answer it |
| 2 | Partially relevant: touches the answer but incomplete |
| 3 | Directly relevant: actually answers the query |

**How the passages get in.** The `state` is a dict: `{"query": ..., "passages": [...]}`. Backtick names in an
instruction refer to keys of that dict, and `` `passages[3]` `` points at **one element of the list**. So we build eight
questions, `c0`..`c7`, each one asking about its own passage, and send them **all in one request**. Jev answers them in
parallel, so eight questions cost one round trip.

**The filter.** `KEEP = 1.5` is halfway between *tangential* (1) and *partially relevant* (2). Anything below it is
closer to "doesn't help" than to "helps", so we drop it.

This cell only **defines** the rubric and the function. Nothing is sent to Jev yet.

In [3]:
LEVELS = [
    "Irrelevant: does not address the query at all",
    "Tangential: same topic area but does not help answer it",
    "Partially relevant: touches the answer but incomplete",
    "Directly relevant: actually answers the query",
]
KEEP = 1.5   # drop anything below "partially relevant"


def rerank(query, candidates, client):
    if "LEVELS" not in globals():
        raise RuntimeError("LEVELS is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    # One Score question per passage: c0 asks about passages[0], c1 about passages[1], ...
    questions = {
        f"c{i}": Score(instructions=f"How well does `passages[{i}]` answer `query`?", criteria=LEVELS)
        for i in range(len(candidates))
    }
    r = client.system_one(state={"query": query, "passages": candidates}, questions=questions)
    scored = [(i, r.scores[f"c{i}"].score) for i in range(len(candidates))]
    return sorted(scored, key=lambda t: t[1], reverse=True), r.usage

## Step 3 · Rerank and filter

One call scores all eight passages. We print them in the **new** order, with the bar showing the score out of 3, the
`keep`/`drop` decision, and where the retriever had originally put each passage.

In [4]:
scored, usage = rerank(QUERY, CANDIDATES, jev)
for rank, (i, s) in enumerate(scored, 1):
    tag = "keep" if s >= KEEP else "drop"
    print(f"{rank}. [{tag}] {bar(s / 3, 16)} {s:.2f}  (was #{i + 1})  {CANDIDATES[i][:50]}")

print(f"\nreranked {len(CANDIDATES)} passages in one request: {usage.input_tokens} tokens, "
      f"${usage.input_tokens / 1e6 * PRICE_PER_M:.6f}")

1. [keep] ████████████████ 2.99  (was #6)  If your starter is sluggish, feed it more often an
2. [drop] █████··········· 0.93  (was #5)  A sourdough starter is a fermented culture of flou
3. [drop] ███············· 0.64  (was #8)  Preheat a Dutch oven to 230C before baking for the
4. [drop] ███············· 0.62  (was #2)  Our starter kit for new bakers includes three loaf
5. [drop] ███············· 0.51  (was #3)  Bread has been a staple food across human civiliza
6. [drop] █··············· 0.12  (was #1)  The sun was rising over the bakery as the queue fo
7. [drop] ················ 0.05  (was #4)  Rising interest rates have pushed up the cost of c
8. [drop] ················ 0.01  (was #7)  Starter homes in the area now average four hundred

reranked 8 passages in one request: 1179 tokens, $0.000050


**What to notice**
- The real answer jumps from **#6 to #1**, with a score at or near the top of the scale (3 = directly relevant).
- The background passage (*"A sourdough starter is a fermented culture..."*) comes second but is **dropped**: it is on
  topic, yet it doesn't tell you what to do. That is exactly the tangential/useful distinction the rubric draws.
- The keyword traps (*sun rising*, *interest rates rising*, *starter homes*) score near **0**. Sharing words with the
  query earns nothing.
- Eight judgments, one request, around a thousand input tokens: a tiny fraction of a cent.

## Step 4 · What the LLM actually gets

Sorting is only half the job. `context_for_llm()` keeps the passages that pass `KEEP` and drops the rest, so the LLM
never sees the distractors. When **nothing** passes, it returns an empty list: the honest move is then to tell the
LLM there is no relevant context (or skip the LLM and say "I don't know") instead of letting it answer from junk.

We try it on the real question and on one the passages can't answer at all.

In [5]:
def context_for_llm(query, candidates, client):
    scored, _ = rerank(query, candidates, client)
    return [candidates[i] for i, s in scored if s >= KEEP]


for q in [QUERY, "How do I fix a flat bike tyre?"]:
    kept = context_for_llm(q, CANDIDATES, jev)
    print(f"query: {q}")
    if kept:
        for passage in kept:
            print(f"   -> {passage}")
    else:
        print("   -> nothing relevant: tell the LLM there is no context, or answer \"I don't know\"")
    print()

query: What should I do if my sourdough starter isn't rising?
   -> If your starter is sluggish, feed it more often and keep it around 24-26C; discard half before each feeding to concentrate the yeast.



query: How do I fix a flat bike tyre?
   -> nothing relevant: tell the LLM there is no context, or answer "I don't know"



**What to notice**
- For the sourdough question, the LLM gets **one** passage, the right one, instead of eight mostly wrong ones.
- For the bike question, every passage is dropped. A plain top-k retriever would still have handed the LLM eight
  passages about bread and house prices; the threshold is what lets the system say *"not in my documents"*.

## Step 5 · Give it a face

Everything from here on is **presentation only**: a small style kit, a text box wired to `rerank()` with `ipywidgets`,
and an HTML before/after view. The reranking logic above is unchanged.

In [6]:
import ipywidgets as widgets
from IPython.display import HTML, display

# small style kit
INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:820px;">{inner}</div>'


def text_app(placeholder, render, button="Run", value=""):
    """A text box + button. On submit, call render(text) and show the HTML it returns."""
    inp = widgets.Text(value=value, placeholder=placeholder, continuous_update=False,
                       layout=widgets.Layout(width="68%"))
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            if inp.value.strip():
                display(HTML(box("<i>thinking...</i>")))
                html = render(inp.value.strip())
                out.clear_output(wait=True)
                display(HTML(html))

    btn.on_click(go)
    inp.observe(go, names="value")   # fires when you press Enter
    display(widgets.HBox([inp, btn]), out)

`rag_html()` shows the retriever's original order on the left and Jev's reranked, filtered order on the right.
Green **KEEP** badges are what would be passed to the LLM.

In [7]:
def rag_html(query):
    scored, _ = rerank(query, CANDIDATES, jev)
    before = "".join(f'<div style="padding:4px 0;color:{SLATE};">{i + 1}. {c[:52]}</div>' for i, c in enumerate(CANDIDATES))
    after = ""
    for i, s in scored:
        keep = s >= KEEP
        col = SAGE if keep else "#B9B4A8"
        after += (f'<div style="padding:4px 0;color:{INK if keep else "#999"};">'
                  f'<span style="{BADGE}background:{col};font-size:11px;">{"KEEP" if keep else "drop"}</span> '
                  f'<b>{s:.2f}</b> &nbsp; {CANDIDATES[i][:46]}</div>')
    return box(f'<div style="font-size:13px;color:{SLATE};margin-bottom:6px;">query: <b>{query}</b></div>'
               f'<div style="display:flex;gap:18px;">'
               f'<div style="flex:1;{CARD}"><b>BEFORE</b> (keyword order){before}</div>'
               f'<div style="flex:1;{CARD}"><b>AFTER</b> (Jev rerank){after}</div></div>')


display(HTML(rag_html(QUERY)))

## Step 6 · Try it yourself

Same 8 passages, your query. Type it and press **Enter** or click **Rerank** (needs a live kernel). Good ones to try:
- `how hot should my oven be?` → the **Dutch oven at 230C** passage jumps to the top and is the only one kept.
- `house prices` → **starter homes** (a trap for the sourdough query) is now the right answer and is kept.
- `How do I fix a flat bike tyre?` → everything scores near 0 and **nothing is kept**.
- Change the query back to the sourdough question and compare.

In [8]:
text_app("a query to rerank the passage set", rag_html, "Rerank", value=QUERY)

Output()

## Recap and exercises

**Key ideas**
1. A reranker is **one `Score` per candidate**, all sent in a single Jev call. Eight passages, one round trip.
2. `` `passages[i]` `` in an instruction points at one list element in the `state`, so each question reads its own passage in context.
3. Calibrated scores let you **filter**, not just sort. A threshold in code decides what the LLM sees.
4. "Nothing passed" is a valid, useful result: it is how a RAG system learns to say *"not in my documents"*.

**Exercises**
- Lower `KEEP` to 0.8 and re-run Step 3. Which passage now gets through, and would you want it in the context?
- Add a ninth candidate that answers the question in different words (e.g. *"Warm the jar and feed twice a day"*) and see where it lands.
- Change the rubric to five levels and adjust `KEEP` to match. Does the ranking change?
- Replace `CANDIDATES` with the top 10 results from a real search over your own docs, and compare top-1 before and after.